# DigiSteel-YOLO v2: Training & Evaluation Pipeline

## WFCA + EMA + GhostConv + Inner-WIoU

**Team:** Hazem Elerefy, Youssef Sherif, Mohamed Salah, Moamen Esmat, Mahmoud Hisham  
**Supervisor:** Dr. Tarek Ghoneimy  
**Deadline:** Sunday June 7, 2026

---

### Pipeline:
1. Setup environment + clone repo
2. Register custom modules (WFCA, EMA, GhostConv)
3. Download NEU-DET dataset
4. Train baseline YOLOv11n (100 epochs, pretrained)
5. Train DigiSteel v2 (100 epochs, from scratch — novel architecture)
6. Evaluate both models (clean + robustness)
7. Comparison table with all metrics

**⚠️ Enable GPU:** Runtime → Change runtime type → **T4 GPU**

## 1. Setup Environment

In [ ]:
#@title 1.1 Install Dependencies
!pip install "ultralytics>=8.3.0,<8.4.0" -q
!pip install opencv-python albumentations numpy pandas matplotlib seaborn tqdm pyyaml -q

import torch
import ultralytics
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
print(f'Ultralytics: {ultralytics.__version__}')

In [ ]:
#@title 1.2 Clone Repository
import os
os.chdir('/content')

REPO_URL = 'https://github.com/hazemelerefey/DigiSteel-YOLO.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

if not os.path.exists('DigiSteel-YOLO'):
    !git clone -b $BRANCH $REPO_URL
else:
    os.chdir('DigiSteel-YOLO')
    !git pull origin $BRANCH
    os.chdir('/content')

os.chdir('DigiSteel-YOLO')
!pip install -e . -q

from pathlib import Path
for d in ['datasets', 'runs', 'evals', 'weights']:
    Path(d).mkdir(exist_ok=True)

print(f'Project dir: {os.getcwd()}')

## 2. Register Custom Modules + Verify

In [ ]:
#@title 2.1 Register WFCA, EMA, GhostConv with Ultralytics
from digisteel.engine.trainer import register_custom_modules
from digisteel.modules import WFCA, EMA, GhostConv, InnerWIoULoss, inner_wiou_iou
from digisteel.perturbations import PerturbationSuite

# Register custom modules so Ultralytics can parse digisteel_v2.yaml
register_custom_modules()

print('✓ Custom modules registered with Ultralytics')
print(f'  WFCA: {WFCA}')
print(f'  EMA: {EMA}')
print(f'  GhostConv: {GhostConv}')
print(f'  InnerWIoULoss: {InnerWIoULoss}')
print(f'  PerturbationSuite: {PerturbationSuite}')

In [ ]:
#@title 2.2 Quick Smoke Test — Build DigiSteel v2 Model
from ultralytics import YOLO
from digisteel.engine.trainer import patch_model_for_digisteel

# Build v2 model and verify modules
model_v2 = YOLO('configs/models/digisteel_v2.yaml')
patch_model_for_digisteel(model_v2)

# Test forward pass
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dummy = torch.randn(1, 3, 640, 640).to(device)
model_v2.model.to(device)
with torch.no_grad():
    out = model_v2.model(dummy)

total_params = sum(p.numel() for p in model_v2.model.parameters())
print(f'\n✓ DigiSteel v2 model built successfully')
print(f'  Parameters: {total_params:,} ({total_params/1e6:.2f}M)')
print(f'  Output shape: {out[0].shape if isinstance(out, tuple) else out.shape}')

## 3. Download NEU-DET Dataset

In [ ]:
#@title 3.1 Configure Kaggle API
from pathlib import Path
import os

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

KAGGLE_USERNAME = 'hazemelerefy'  #@param {type:"string"}
KAGGLE_KEY = 'KGAT_33e87e0d22cbd7faa4e69b53eb191732'  #@param {type:"string"}

kaggle_json = kaggle_dir / 'kaggle.json'
kaggle_json.write_text(f'{{"username":"{KAGGLE_USERNAME}","key":"{KAGGLE_KEY}"}}')
kaggle_json.chmod(0o600)

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

print(f'✓ Kaggle credentials configured for: {KAGGLE_USERNAME}')

In [ ]:
#@title 3.2 Download NEU-DET from Kaggle
from pathlib import Path

NEU_DET_DATASET = 'sovitrath/neu-steel-surface-defect-detect-trainvalid-split'  #@param {type:"string"}

neu_dir = Path('datasets/NEU-DET')

if not neu_dir.exists() or len(list(neu_dir.glob('**/*.jpg'))) < 100:
    print('Downloading NEU-DET...')
    !kaggle datasets download -d $NEU_DET_DATASET -p datasets/NEU-DET --unzip -q
    print('✓ Downloaded')
else:
    print('✓ NEU-DET already exists')

jpg_count = len(list(neu_dir.glob('**/*.jpg')))
print(f'  Total images: {jpg_count}')

In [ ]:
#@title 3.3 Prepare Dataset in YOLO Format
import random
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

NEU_DET_CLASSES = {
    'crazing': 0, 'inclusion': 1, 'patches': 2,
    'pitted_surface': 3, 'rolled-in_scale': 4, 'scratches': 5
}

def convert_voc_to_yolo(xml_path, img_w=200, img_h=200):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in NEU_DET_CLASSES:
            continue
        cid = NEU_DET_CLASSES[name]
        bb = obj.find('bndbox')
        xmin = float(bb.find('xmin').text)
        ymin = float(bb.find('ymin').text)
        xmax = float(bb.find('xmax').text)
        ymax = float(bb.find('ymax').text)
        cx = ((xmin + xmax) / 2) / img_w
        cy = ((ymin + ymax) / 2) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        lines.append(f'{cid} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return '\n'.join(lines)

yolo_dir = neu_dir / 'yolo'
if yolo_dir.exists() and len(list((yolo_dir / 'images' / 'train').glob('*.jpg'))) > 100:
    print('✓ YOLO format dataset already exists')
    for split in ['train', 'val', 'test']:
        n = len(list((yolo_dir / 'images' / split).glob('*.jpg')))
        print(f'  {split}: {n} images')
else:
    print('Converting to YOLO format...')
    for split in ['train', 'val', 'test']:
        (yolo_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
        (yolo_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

    train_imgs = list((neu_dir / 'train_images').glob('*.jpg'))
    valid_imgs = list((neu_dir / 'valid_images').glob('*.jpg'))
    all_imgs = train_imgs + valid_imgs
    ann_dirs = [neu_dir / 'train_annotations'] * len(train_imgs) + [neu_dir / 'valid_annotations'] * len(valid_imgs)

    rng = random.Random(42)
    converted = 0
    for img_path, ann_dir in zip(all_imgs, ann_dirs):
        xml_path = ann_dir / (img_path.stem + '.xml')
        if not xml_path.exists():
            continue
        yolo_text = convert_voc_to_yolo(xml_path)

        r = rng.random()
        split = 'train' if r < 0.7 else ('val' if r < 0.9 else 'test')

        shutil.copy2(img_path, yolo_dir / 'images' / split / img_path.name)
        (yolo_dir / 'labels' / split / (img_path.stem + '.txt')).write_text(yolo_text)
        converted += 1

    print(f'✓ Converted {converted} images')
    for split in ['train', 'val', 'test']:
        n = len(list((yolo_dir / 'images' / split).glob('*.jpg')))
        print(f'  {split}: {n} images')

## 4. Train Baseline YOLOv11n (pretrained)

In [ ]:
#@title 4.1 Train YOLOv11n Baseline (100 epochs)
from ultralytics import YOLO
import time

EPOCHS = 100  #@param {type:"integer"}
IMG_SIZE = 640  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

CONFIG_PATH = 'configs/data/neu_det.yaml'
BASELINE_NAME = f'baseline_neu_det_seed{SEED}'

print('='*60)
print('  Training YOLOv11n Baseline (pretrained)')
print('='*60)
print(f'  Epochs: {EPOCHS}')
print(f'  Image: {IMG_SIZE}')
print(f'  Batch: {BATCH_SIZE}')
print(f'  Init: Pretrained (yolo11n.pt)')
print('='*60)

baseline_model = YOLO('yolo11n.pt')
start = time.time()

baseline_model.train(
    data=CONFIG_PATH,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    seed=SEED,
    project='runs',
    name=BASELINE_NAME,
    exist_ok=True,
    patience=30,
    verbose=True,
)

elapsed = time.time() - start
print(f'\n✓ Baseline training complete in {elapsed/3600:.1f} hours')
print(f'  Best weights: runs/{BASELINE_NAME}/weights/best.pt')

In [ ]:
#@title 4.2 Evaluate Baseline + Measure FPS
from ultralytics import YOLO
from pathlib import Path
import time

baseline_path = f'runs/{BASELINE_NAME}/weights/best.pt'

if Path(baseline_path).exists():
    bmodel = YOLO(baseline_path)
    val = bmodel.val(data=CONFIG_PATH, imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=True)

    # FPS benchmark
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    bmodel.model.to(device).eval()
    # Warmup
    for _ in range(10):
        with torch.no_grad():
            bmodel.model(dummy)
    # Measure
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    N = 100
    for _ in range(N):
        with torch.no_grad():
            bmodel.model(dummy)
    if device == 'cuda':
        torch.cuda.synchronize()
    fps = N / (time.time() - t0)

    baseline_params = sum(p.numel() for p in bmodel.model.parameters())
    print('\n' + '='*60)
    print('  Baseline Results')
    print('='*60)
    print(f'  mAP@0.5:      {val.box.map50:.4f}')
    print(f'  mAP@0.5:0.95: {val.box.map:.4f}')
    print(f'  Precision:    {val.box.mp:.4f}')
    print(f'  Recall:       {val.box.mr:.4f}')
    print(f'  Params:       {baseline_params:,} ({baseline_params/1e6:.2f}M)')
    print(f'  FPS:          {fps:.1f}')

    classes = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
    print('\n  Per-class mAP@0.5:')
    for name, ap in zip(classes, val.box.ap50):
        print(f'    {name:<20} {ap:.4f}')

    # Save metrics for comparison
    import json
    Path('evals').mkdir(exist_ok=True)
    with open('evals/baseline_metrics.json', 'w') as f:
        json.dump({
            'model': 'YOLOv11n Baseline',
            'mAP50': val.box.map50,
            'mAP50_95': val.box.map,
            'precision': val.box.mp,
            'recall': val.box.mr,
            'params': baseline_params,
            'fps': fps,
            'per_class_ap50': {name: float(ap) for name, ap in zip(classes, val.box.ap50)},
        }, f, indent=2)
    print('\n✓ Metrics saved to evals/baseline_metrics.json')
else:
    print(f'⚠ Baseline not found: {baseline_path}')
    print('  Run training cell first.')

## 5. Train DigiSteel v2 (from scratch)

In [ ]:
#@title 5.1 Train DigiSteel v2 (100 epochs) with Inner-WIoU
from digisteel.engine.trainer import DigiSteelTrainer, register_custom_modules
import time

# MUST register before loading YAML
register_custom_modules()

EPOCHS = 100  #@param {type:"integer"}
IMG_SIZE = 640  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
SEED = 42  #@param {type:"integer"}

V2_NAME = f'digisteel_v2_neu_det_seed{SEED}'

print('='*60)
print('  Training DigiSteel v2')
print('='*60)
print(f'  Architecture: GhostConv + WFCA + EMA')
print(f'  Loss: Inner-WIoU (lambda=0.5)')
print(f'  Epochs: {EPOCHS}')
print(f'  Image: {IMG_SIZE}')
print(f'  Batch: {BATCH_SIZE}')
print(f'  Init: Random (novel architecture)')
print('='*60)

start = time.time()

# Use DigiSteelTrainer to get Inner-WIoU loss integration
trainer = DigiSteelTrainer.create_trainer(overrides={
    'model': 'configs/models/digisteel_v2.yaml',
    'data': CONFIG_PATH,
    'epochs': EPOCHS,
    'imgsz': IMG_SIZE,
    'batch': BATCH_SIZE,
    'seed': SEED,
    'project': 'runs',
    'name': V2_NAME,
    'exist_ok': True,
    'patience': 30,
    'verbose': True,
})

trainer.train()

elapsed = time.time() - start
print(f'\n✓ DigiSteel v2 training complete in {elapsed/3600:.1f} hours')
print(f'  Best weights: runs/{V2_NAME}/weights/best.pt')

In [ ]:
#@title 5.2 Evaluate DigiSteel v2 + Measure FPS
from ultralytics import YOLO
from digisteel.engine.trainer import register_custom_modules
from pathlib import Path
import time

register_custom_modules()

v2_path = f'runs/{V2_NAME}/weights/best.pt'

if Path(v2_path).exists():
    v2model = YOLO(v2_path)
    val = v2model.val(data=CONFIG_PATH, imgsz=IMG_SIZE, batch=BATCH_SIZE, verbose=True)

    # FPS benchmark
    import torch
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
    v2model.model.to(device).eval()
    for _ in range(10):
        with torch.no_grad():
            v2model.model(dummy)
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.time()
    N = 100
    for _ in range(N):
        with torch.no_grad():
            v2model.model(dummy)
    if device == 'cuda':
        torch.cuda.synchronize()
    fps = N / (time.time() - t0)

    v2_params = sum(p.numel() for p in v2model.model.parameters())
    print('\n' + '='*60)
    print('  DigiSteel v2 Results')
    print('='*60)
    print(f'  mAP@0.5:      {val.box.map50:.4f}')
    print(f'  mAP@0.5:0.95: {val.box.map:.4f}')
    print(f'  Precision:    {val.box.mp:.4f}')
    print(f'  Recall:       {val.box.mr:.4f}')
    print(f'  Params:       {v2_params:,} ({v2_params/1e6:.2f}M)')
    print(f'  FPS:          {fps:.1f}')

    classes = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
    print('\n  Per-class mAP@0.5:')
    for name, ap in zip(classes, val.box.ap50):
        print(f'    {name:<20} {ap:.4f}')

    import json
    with open('evals/v2_metrics.json', 'w') as f:
        json.dump({
            'model': 'DigiSteel v2',
            'mAP50': val.box.map50,
            'mAP50_95': val.box.map,
            'precision': val.box.mp,
            'recall': val.box.mr,
            'params': v2_params,
            'fps': fps,
            'per_class_ap50': {name: float(ap) for name, ap in zip(classes, val.box.ap50)},
        }, f, indent=2)
    print('\n✓ Metrics saved to evals/v2_metrics.json')
else:
    print(f'⚠ DigiSteel v2 not found: {v2_path}')
    print('  Run training cell first.')

## 6. Robustness Evaluation

In [ ]:
#@title 6.1 Run Robustness Sweep (6 types x 4 levels = 24 points)
import cv2
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from digisteel.perturbations import PerturbationSuite
from digisteel.engine.trainer import register_custom_modules

register_custom_modules()

def evaluate_robust(model_path, data_yaml, imgsz=640):
    """Evaluate model on clean + 24 perturbation points.

    Uses safe temp directories — original validation images are NEVER modified.
    """
    model = YOLO(model_path)
    suite = PerturbationSuite()
    val_img_dir = Path('datasets/NEU-DET/yolo/images/val')

    # Create a safe working copy of validation images
    work_dir = Path('/tmp/val_work')
    if work_dir.exists():
        shutil.rmtree(work_dir)
    work_dir.mkdir(parents=True)
    for img in val_img_dir.glob('*.jpg'):
        shutil.copy2(img, work_dir / img.name)

    # We need a temp data yaml that points to our working directory
    import yaml
    with open(data_yaml) as f:
        data_cfg = yaml.safe_load(f)

    # Clean evaluation (on original images)
    clean_val = model.val(data=data_yaml, imgsz=imgsz, verbose=False)
    clean_map = clean_val.box.map50
    results = [{'perturbation': 'clean', 'level': 0, 'mAP50': clean_map}]
    print(f'  Clean: mAP50={clean_map:.4f}')

    # Perturbed evaluation
    for config in suite.all_configs():
        # Apply perturbation to working copy
        for img in work_dir.glob('*.jpg'):
            orig = cv2.imread(str(img))
            if orig is None:
                continue
            degraded = suite.apply(orig, config.name, config.level)
            cv2.imwrite(str(val_img_dir / img.name), degraded)

        # Evaluate on perturbed images
        val = model.val(data=data_yaml, imgsz=imgsz, verbose=False)
        results.append({
            'perturbation': config.name,
            'level': config.level,
            'mAP50': val.box.map50
        })
        print(f'  {config.name}_L{config.level}: mAP50={val.box.map50:.4f}')

    # ALWAYS restore originals from working copy
    for img in work_dir.glob('*.jpg'):
        shutil.copy2(img, val_img_dir / img.name)
    print('  ✓ Original validation images restored')

    return pd.DataFrame(results)

# Run on both models
baseline_weights = f'runs/{BASELINE_NAME}/weights/best.pt'
v2_weights = f'runs/{V2_NAME}/weights/best.pt'

if Path(baseline_weights).exists():
    print('\nEvaluating baseline robustness...')
    baseline_robust = evaluate_robust(baseline_weights, CONFIG_PATH)
    baseline_robust.to_csv('evals/baseline_robustness.csv', index=False)
    print('✓ Baseline robustness saved to evals/baseline_robustness.csv')
else:
    print(f'⚠ Baseline not found: {baseline_weights}')
    baseline_robust = None

if Path(v2_weights).exists():
    print('\nEvaluating DigiSteel v2 robustness...')
    v2_robust = evaluate_robust(v2_weights, CONFIG_PATH)
    v2_robust.to_csv('evals/digisteel_v2_robustness.csv', index=False)
    print('✓ DigiSteel v2 robustness saved to evals/digisteel_v2_robustness.csv')
else:
    print(f'⚠ DigiSteel v2 not found: {v2_weights}')
    v2_robust = None

## 7. Comparison Tables

In [ ]:
#@title 7.1 Baseline vs DigiSteel v2 — Full Comparison
import json
import pandas as pd
from pathlib import Path

print('='*80)
print('  DigiSteel v2 vs Baseline — Final Comparison')
print('='*80)

# Load saved metrics
b_metrics_path = Path('evals/baseline_metrics.json')
v_metrics_path = Path('evals/v2_metrics.json')

if b_metrics_path.exists() and v_metrics_path.exists():
    with open(b_metrics_path) as f:
        bm = json.load(f)
    with open(v_metrics_path) as f:
        vm = json.load(f)

    print(f'\n{'Metric':<30} {'Baseline':>12} {'DigiSteel v2':>12} {'Delta':>10}')
    print('-'*68)
    print(f"{'mAP@0.5':<30} {bm['mAP50']:>12.4f} {vm['mAP50']:>12.4f} {vm['mAP50']-bm['mAP50']:>+10.4f}")
    print(f"{'mAP@0.5:0.95':<30} {bm['mAP50_95']:>12.4f} {vm['mAP50_95']:>12.4f} {vm['mAP50_95']-bm['mAP50_95']:>+10.4f}")
    print(f"{'Precision':<30} {bm['precision']:>12.4f} {vm['precision']:>12.4f} {vm['precision']-bm['precision']:>+10.4f}")
    print(f"{'Recall':<30} {bm['recall']:>12.4f} {vm['recall']:>12.4f} {vm['recall']-bm['recall']:>+10.4f}")
    print(f"{'Params (M)':<30} {bm['params']/1e6:>12.2f} {vm['params']/1e6:>12.2f} {vm['params']/1e6-bm['params']/1e6:>+10.2f}")
    print(f"{'FPS':<30} {bm['fps']:>12.1f} {vm['fps']:>12.1f} {vm['fps']-bm['fps']:>+10.1f}")

    # Param efficiency
    eff_b = bm['mAP50'] / (bm['params'] / 1e6)
    eff_v = vm['mAP50'] / (vm['params'] / 1e6)
    print(f"{'Param Efficiency (mAP/M)':<30} {eff_b:>12.4f} {eff_v:>12.4f} {eff_v-eff_b:>+10.4f}")

    # Per-class comparison
    classes = list(bm['per_class_ap50'].keys())
    print(f'\n{'Class':<25} {'Baseline':>10} {'v2':>10} {'Delta':>10}')
    print('-'*58)
    for cls in classes:
        b_ap = bm['per_class_ap50'][cls]
        v_ap = vm['per_class_ap50'][cls]
        print(f"{cls:<25} {b_ap:>10.4f} {v_ap:>10.4f} {v_ap-b_ap:>+10.4f}")

    # Robustness comparison
    b_csv = Path('evals/baseline_robustness.csv')
    v_csv = Path('evals/digisteel_v2_robustness.csv')
    if b_csv.exists() and v_csv.exists():
        b_rob = pd.read_csv(b_csv)
        v_rob = pd.read_csv(v_csv)

        clean_b = b_rob[b_rob['perturbation'] == 'clean']['mAP50'].values[0]
        clean_v = v_rob[v_rob['perturbation'] == 'clean']['mAP50'].values[0]
        avg_b = b_rob[b_rob['perturbation'] != 'clean']['mAP50'].mean()
        avg_v = v_rob[v_rob['perturbation'] != 'clean']['mAP50'].mean()
        stab_b = avg_b / clean_b if clean_b > 0 else 0
        stab_v = avg_v / clean_v if clean_v > 0 else 0

        print(f'\n{'Robustness Metric':<30} {'Baseline':>12} {'DigiSteel v2':>12} {'Delta':>10}')
        print('-'*68)
        print(f"{'Clean mAP@0.5':<30} {clean_b:>12.4f} {clean_v:>12.4f} {clean_v-clean_b:>+10.4f}")
        print(f"{'Avg Perturbed mAP@0.5':<30} {avg_b:>12.4f} {avg_v:>12.4f} {avg_v-avg_b:>+10.4f}")
        print(f"{'Stability Ratio':<30} {stab_b:>12.4f} {stab_v:>12.4f} {stab_v-stab_b:>+10.4f}")

        # Per-perturbation breakdown
        print(f'\n{'Perturbation':<25} {'Level':>6} {'Baseline':>10} {'v2':>10} {'Delta':>10}')
        print('-'*65)
        for _, row in b_rob[b_rob['perturbation'] != 'clean'].iterrows():
            vrow = v_rob[(v_rob['perturbation'] == row['perturbation']) & (v_rob['level'] == row['level'])]
            if len(vrow) > 0:
                vm_val = vrow['mAP50'].values[0]
                delta = vm_val - row['mAP50']
                print(f"{row['perturbation']:<25} {int(row['level']):>6} {row['mAP50']:>10.4f} {vm_val:>10.4f} {delta:>+10.4f}")
else:
    print('⚠ Metrics not found. Run cells 4.2 and 5.2 first.')
    print(f'  Expected: {b_metrics_path}, {v_metrics_path}')

In [ ]:
#@title 7.2 Reference Papers Comparison
import json
from pathlib import Path

print('='*100)
print('  DigiSteel v2 vs Reference Papers (NEU-DET)')
print('='*100)

refs = [
    ('P02-LAM-YOLOv10n',      'YOLOv10n', 94.39, None,  154),
    ('P10-KDM-YOLO',          'YOLOv10n', 95.40, 3.29,  155.6),
    ('P11-YOLOv11-EMD',       'YOLOv11',  94.90, None,  None),
    ('P03-YOLO-LSDI',         'YOLOv11n', 83.00, 2.70,  162.1),
    ('P07-ASFRW-YOLO',        'YOLOv5s',  83.20, 6.20,  125),
    ('P09-EFEN-YOLOv8',       'YOLOv8',   80.40, None,  None),
    ('P08-MSFE-YOLO',         'YOLOv11s', 79.80, 11.69, 89.3),
    ('P04-Lightweight-YOLOv8', 'YOLOv8',  78.60, 2.04,  171.5),
    ('P05-SCCI-YOLO',         'YOLOv8n',  78.60, 1.68,  270.2),
]

print(f"{'Model':<28} {'Base':<12} {'mAP@0.5':>10} {'Params(M)':>10} {'FPS':>8}")
print('-'*72)
for name, base, m, p, fps in refs:
    ps = f'{p:.2f}' if p else 'N/A'
    fs = f'{fps:.1f}' if fps else 'N/A'
    print(f'{name:<28} {base:<12} {m:>10.2f} {ps:>10} {fs:>8}')

print('-'*72)

# Load actual results
b_path = Path('evals/baseline_metrics.json')
v_path = Path('evals/v2_metrics.json')

if b_path.exists():
    with open(b_path) as f:
        bm = json.load(f)
    print(f"{'YOLOv11n Baseline (ours)':<28} {'YOLOv11n':<12} {bm['mAP50']*100:>10.2f} {bm['params']/1e6:>10.2f} {bm['fps']:>8.1f}")
else:
    print(f"{'YOLOv11n Baseline (ours)':<28} {'YOLOv11n':<12} {'???':>10} {'2.59':>10} {'???':>8}")
    print('  ⚠ Run cell 4.2 to get actual results')

if v_path.exists():
    with open(v_path) as f:
        vm = json.load(f)
    print(f"{'DigiSteel v2 (ours)':<28} {'YOLOv11n':<12} {vm['mAP50']*100:>10.2f} {vm['params']/1e6:>10.2f} {vm['fps']:>8.1f}")
else:
    print(f"{'DigiSteel v2 (ours)':<28} {'YOLOv11n':<12} {'???':>10} {'~2.5':>10} {'???':>8}")
    print('  ⚠ Run cell 5.2 to get actual results')

print('='*72)
print('\nNote: Baseline uses pretrained weights; DigiSteel v2 trains from scratch.')
print('This reflects real-world conditions: novel architectures lack pretrained weights.')

## 8. Save Results

In [ ]:
#@title 8.1 Download All Results
from google.colab import files
from pathlib import Path

files_to_zip = []

# Model weights
for run_dir in Path('runs').glob('*'):
    best = run_dir / 'weights' / 'best.pt'
    if best.exists():
        files_to_zip.append(str(best))
        print(f'  ✓ {best}')

# Eval files
for f in Path('evals').glob('*'):
    if f.is_file():
        files_to_zip.append(str(f))
        print(f'  ✓ {f}')

if files_to_zip:
    import subprocess
    subprocess.run(['zip', '-r', 'digisteel_v2_results.zip'] + files_to_zip, check=True)
    print(f'\n✓ Packaged {len(files_to_zip)} files')
    files.download('digisteel_v2_results.zip')
else:
    print('⚠ No results to download. Run training first.')

---

## Notes

- **If session times out:** Re-run cells 1-2 (setup), then skip to the interrupted training cell
- **100 epochs ≈ 1-2 hours** on T4 GPU per model
- **Total time ≈ 3-4 hours** for both models + evaluation
- **Inner-WIoU** is integrated via `DigiSteelTrainer` which patches `BboxLoss.iou()` with `InnerWIoUAdapter`
- **Fair comparison note:** Baseline uses pretrained `yolo11n.pt`; v2 trains from scratch because it's a novel architecture without pretrained weights. This matches real-world conditions.
- **Custom module registration** must happen before loading the v2 YAML — always run cell 2.1 first
- **Robustness eval** uses safe temp directories — original validation images are never permanently modified